In [1]:
import os
import joblib
import numpy as np
import pandas as pd

from tensorflow import keras
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
# Paths to saved artifacts
ann_path = "../artifacts/cvd/tuned_ann.keras"
gb_path = "../artifacts/cvd/tuned_gradient_boosting.pkl"

X_test_path = "../artifacts/cvd/X_test_scaled.npy"
y_test_path = "../artifacts/cvd/y_test.npy"

# Load models
final_ann_model = keras.models.load_model(ann_path)
final_gb_model = joblib.load(gb_path)

# Load test data
X_test_scaled = np.load(X_test_path)
y_test = np.load(y_test_path)

print("Models and test data loaded successfully.")
print("X_test shape:", X_test_scaled.shape)
print("y_test shape:", y_test.shape)

Models and test data loaded successfully.
X_test shape: (13740, 11)
y_test shape: (13740,)


In [3]:
# ANN predictions
ann_probabilities = final_ann_model.predict(X_test_scaled, verbose=0).ravel()
ann_predictions = (ann_probabilities >= 0.5).astype(int)

# Gradient Boosting predictions
gb_probabilities = final_gb_model.predict_proba(X_test_scaled)[:, 1]
gb_predictions = (gb_probabilities >= 0.5).astype(int)

print("Predictions generated successfully.")

print("\nANN:")
print("Prediction shape:", ann_predictions.shape)
print("First 10 predicted classes:", ann_predictions[:10])

print("\nGradient Boosting:")
print("Prediction shape:", gb_predictions.shape)
print("First 10 predicted classes:", gb_predictions[:10])

Predictions generated successfully.

ANN:
Prediction shape: (13740,)
First 10 predicted classes: [0 0 1 1 1 0 1 0 0 0]

Gradient Boosting:
Prediction shape: (13740,)
First 10 predicted classes: [0 0 1 1 1 0 1 0 0 1]


In [4]:
# Function to calculate evaluation metrics
def calculate_metrics(y_true, y_pred, y_prob):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred),
        "Recall": recall_score(y_true, y_pred),
        "Specificity": tn / (tn + fp),
        "F1 Score": f1_score(y_true, y_pred),
        "ROC-AUC": roc_auc_score(y_true, y_prob)
    }


# Calculate metrics
ann_metrics = calculate_metrics(
    y_test,
    ann_predictions,
    ann_probabilities
)

gb_metrics = calculate_metrics(
    y_test,
    gb_predictions,
    gb_probabilities
)


# Create comparison table
comparison = pd.DataFrame(
    [ann_metrics, gb_metrics],
    index=["Tuned ANN", "Tuned Gradient Boosting"]
)

# Display percentages for easier reading
comparison_display = comparison.copy()

for column in ["Accuracy", "Precision", "Recall", "Specificity", "F1 Score"]:
    comparison_display[column] = (
        comparison_display[column] * 100
    ).round(2)

comparison_display["ROC-AUC"] = comparison_display["ROC-AUC"].round(4)

print("Final Model Comparison:")
display(comparison_display)

Final Model Comparison:


,Accuracy,Precision,Recall,Specificity,F1 Score,ROC-AUC
Tuned ANN,74.00,75.54,70.19,77.74,72.77,0.8075
Tuned Gradient Boosting,73.88,75.55,69.80,77.87,72.56,0.8078


In [5]:
import json

final_model_info = {
    "selected_model": "Tuned ANN",
    "model_file": "tuned_ann.keras",
    "reason": "Best overall balance of accuracy, recall, and F1 score. ROC-AUC difference from tuned Gradient Boosting is negligible.",
    "metrics": {
        "accuracy": 0.7400,
        "precision": 0.7554,
        "recall": 0.7019,
        "specificity": 0.7774,
        "f1_score": 0.7277,
        "roc_auc": 0.8075
    },
    "benchmark_model": "Tuned Gradient Boosting",
    "benchmark_model_file": "tuned_gradient_boosting.pkl"
}

output_path = "../artifacts/cvd/final_model_selection.json"

with open(output_path, "w") as f:
    json.dump(final_model_info, f, indent=4)

print("Final model selection saved successfully.")
print("Selected model:", final_model_info["selected_model"])
print("Saved to:", output_path)

Final model selection saved successfully.
Selected model: Tuned ANN
Saved to: ../artifacts/cvd/final_model_selection.json
